# DBSCAN: Density-Based Spatial Clustering of Applications with Noise

**INDE 577 / CMOR 438 — Rice University**  
**Instructor:** Randy R. Davila, PhD

---

## Overview

DBSCAN is a **density-based** clustering algorithm that groups together closely packed points and marks points in low-density regions as **noise/outliers**. Unlike K-Means, it does not require specifying the number of clusters and can find **arbitrarily shaped** clusters.

## Mathematical Background

### Core Concepts

**$\varepsilon$-neighborhood** of a point $\mathbf{p}$:
$$N_\varepsilon(\mathbf{p}) = \{\mathbf{q} \in D \mid d(\mathbf{p}, \mathbf{q}) \leq \varepsilon\}$$

**Core point**: A point $\mathbf{p}$ is a core point if:
$$|N_\varepsilon(\mathbf{p})| \geq \text{MinPts}$$

**Border point**: A non-core point within $\varepsilon$ of a core point.

**Noise point (outlier)**: Neither a core nor a border point.

### Density-Reachability

Point $\mathbf{q}$ is **directly density-reachable** from $\mathbf{p}$ if $\mathbf{q} \in N_\varepsilon(\mathbf{p})$ and $\mathbf{p}$ is a core point.

Point $\mathbf{q}$ is **density-reachable** from $\mathbf{p}$ if there exists a chain:
$$\mathbf{p} = \mathbf{p}_1, \mathbf{p}_2, \ldots, \mathbf{p}_n = \mathbf{q}$$
where each $\mathbf{p}_{i+1}$ is directly density-reachable from $\mathbf{p}_i$.

### Algorithm
1. For each unvisited point, find its $\varepsilon$-neighborhood
2. If core point: create/expand a cluster by adding all density-reachable points
3. Otherwise: mark as noise (may later be reassigned as border point)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, make_moons, make_circles
from sklearn.cluster import DBSCAN as SklearnDBSCAN
from sklearn.preprocessing import StandardScaler as SklearnScaler
import warnings
warnings.filterwarnings('ignore')

from rice_ml import DBSCAN
from rice_ml.processing.preprocessing import StandardScaler

print("Libraries loaded!")
np.random.seed(42)

## 1. Where K-Means Fails — DBSCAN Succeeds

In [ ]:
from rice_ml import KMeans

# Generate challenging datasets for K-Means
datasets = [
    ('Moons', *make_moons(n_samples=300, noise=0.07, random_state=42)),
    ('Circles', *make_circles(n_samples=300, noise=0.05, factor=0.4, random_state=42)),
    ('Blobs with Noise', *make_blobs(n_samples=300, centers=3, random_state=42)),
]

# Add noise to blobs
X_blobs, y_blobs = datasets[2][1], datasets[2][2]
noise = np.random.uniform(-8, 8, (40, 2))
X_blobs_noise = np.vstack([X_blobs, noise])
datasets[2] = ('Blobs with Noise', X_blobs_noise, None)

dbscan_params = [
    {'eps': 0.2, 'min_samples': 5},
    {'eps': 0.2, 'min_samples': 5},
    {'eps': 0.8, 'min_samples': 5},
]

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
colors_map = plt.cm.tab10(np.linspace(0, 0.8, 10))

for col_idx, ((name, X, _), params) in enumerate(zip(datasets, dbscan_params)):
    scaler = StandardScaler()
    X_s = scaler.fit_transform(X)
    
    # K-Means
    km = KMeans(k=2, random_state=42)
    km_labels = km.fit_predict(X_s)
    
    ax_km = axes[0, col_idx]
    scatter_colors = [colors_map[l] if l >= 0 else 'black' for l in km_labels]
    ax_km.scatter(X_s[:, 0], X_s[:, 1], c=scatter_colors, s=20, alpha=0.8, edgecolors='k', lw=0.2)
    ax_km.set_title(f'K-Means on {name}', fontsize=12, fontweight='bold')
    ax_km.set_xlabel('Feature 1'); ax_km.set_ylabel('Feature 2')
    
    # DBSCAN
    dbscan = DBSCAN(eps=params['eps'], min_samples=params['min_samples'])
    dbscan.fit(X_s)
    labels = dbscan.labels_
    
    ax_db = axes[1, col_idx]
    unique_labels = sorted(set(labels))
    for lbl in unique_labels:
        mask = labels == lbl
        if lbl == -1:
            ax_db.scatter(X_s[mask, 0], X_s[mask, 1], c='black', s=15, alpha=0.5,
                          marker='x', linewidths=1, label='Noise', zorder=3)
        else:
            ax_db.scatter(X_s[mask, 0], X_s[mask, 1], c=[colors_map[lbl % 10]], s=25,
                          alpha=0.8, edgecolors='k', lw=0.2, label=f'Cluster {lbl}')
    
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    n_noise = list(labels).count(-1)
    ax_db.set_title(f'DBSCAN on {name}\n({n_clusters} clusters, {n_noise} noise)',
                    fontsize=12, fontweight='bold')
    ax_db.set_xlabel('Feature 1'); ax_db.set_ylabel('Feature 2')

axes[0, 0].set_ylabel('K-Means', fontsize=13, fontweight='bold')
axes[1, 0].set_ylabel('DBSCAN', fontsize=13, fontweight='bold')
plt.suptitle('K-Means vs DBSCAN on Challenging Datasets', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('dbscan_vs_kmeans.png', dpi=120, bbox_inches='tight')
plt.show()

## 2. Core Points, Border Points, and Noise

In [ ]:
# Small dataset to illustrate point types
np.random.seed(42)
X_demo = np.vstack([
    np.random.randn(50, 2) * 0.5 + [0, 0],
    np.random.randn(50, 2) * 0.5 + [3, 3],
    np.array([[6, 0], [7, 1], [6.5, -1]])  # noise points
])

dbscan_demo = DBSCAN(eps=0.8, min_samples=5)
dbscan_demo.fit(X_demo)
labels_demo = dbscan_demo.labels_
core_idx = dbscan_demo.core_sample_indices_

# Classify each point
is_core = np.zeros(len(X_demo), dtype=bool)
is_core[core_idx] = True
is_noise = labels_demo == -1
is_border = ~is_core & ~is_noise

print(f"Core points: {is_core.sum()}")
print(f"Border points: {is_border.sum()}")
print(f"Noise points: {is_noise.sum()}")
print(f"Clusters found: {len(set(labels_demo)) - (1 if -1 in labels_demo else 0)}")

fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(X_demo[is_core, 0], X_demo[is_core, 1], c='steelblue', s=60,
           alpha=0.9, edgecolors='k', lw=0.5, zorder=3, label=f'Core ({is_core.sum()})')
ax.scatter(X_demo[is_border, 0], X_demo[is_border, 1], c='orange', s=60,
           alpha=0.9, edgecolors='k', lw=0.5, zorder=3, label=f'Border ({is_border.sum()})')
ax.scatter(X_demo[is_noise, 0], X_demo[is_noise, 1], c='red', s=80,
           alpha=0.9, marker='x', linewidths=2, zorder=4, label=f'Noise ({is_noise.sum()})')

# Draw ε-circles for a few core points
for i in core_idx[:3]:
    circle = plt.Circle(X_demo[i], 0.8, fill=False, color='gray', alpha=0.4, linewidth=1)
    ax.add_patch(circle)

ax.set_xlabel('Feature 1', fontsize=11)
ax.set_ylabel('Feature 2', fontsize=11)
ax.set_title('DBSCAN Point Types (ε=0.8, MinPts=5)', fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('dbscan_points.png', dpi=120, bbox_inches='tight')
plt.show()

## 3. Parameter Sensitivity (ε and MinPts)

In [ ]:
X_moons, _ = make_moons(n_samples=300, noise=0.07, random_state=42)
scaler = StandardScaler()
X_moons_s = scaler.fit_transform(X_moons)

eps_vals = [0.1, 0.2, 0.4, 0.7]
fig, axes = plt.subplots(1, 4, figsize=(20, 4))

for ax, eps in zip(axes, eps_vals):
    db = DBSCAN(eps=eps, min_samples=5)
    db.fit(X_moons_s)
    labels = db.labels_
    
    unique_labels = sorted(set(labels))
    n_clusters = len(unique_labels) - (1 if -1 in unique_labels else 0)
    n_noise = list(labels).count(-1)
    
    for lbl in unique_labels:
        mask = labels == lbl
        if lbl == -1:
            ax.scatter(X_moons_s[mask, 0], X_moons_s[mask, 1], c='black', s=15,
                       marker='x', linewidths=1, alpha=0.6)
        else:
            ax.scatter(X_moons_s[mask, 0], X_moons_s[mask, 1],
                       c=[colors_map[lbl % 10]], s=20, alpha=0.8, edgecolors='k', lw=0.2)
    
    ax.set_title(f'ε={eps}\n{n_clusters} clusters, {n_noise} noise', fontsize=11, fontweight='bold')
    ax.set_xlabel('Feature 1')

plt.suptitle('DBSCAN Sensitivity to ε (MinPts=5, Moons Dataset)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('dbscan_eps.png', dpi=120, bbox_inches='tight')
plt.show()

## Summary

| Property | K-Means | DBSCAN |
|---|---|---|
| # Clusters | Must specify $k$ | Automatic |
| Cluster shape | Convex (spherical) | Arbitrary |
| Outliers | None (assigns all points) | Explicit noise label (-1) |
| Hyperparameters | $k$ | $\varepsilon$, MinPts |
| Scalability | $O(nkd)$ | $O(n^2)$ naïve, $O(n \log n)$ with index |

**Key Takeaways:**
- DBSCAN excels at finding **non-convex clusters** and identifying **outliers**
- No need to specify the number of clusters in advance
- Sensitive to $\varepsilon$ — use the **k-distance elbow plot** to choose $\varepsilon$
- Fails when clusters have **very different densities**